In [30]:
import pandas as pd
import requests
import calendar
import matplotlib.pyplot as plt

In [5]:
url = 'https://www.hockey-reference.com/leagues/NHL_2026_games.html'
tables = pd.read_html(requests.get(url).text)

In [6]:
df = tables[0]

In [7]:
df.columns = [
    'game_date'
    ,'game_time'
    ,'team_away'
    ,'team_away_goals'
    ,'team_home'
    ,'team_home_goals'
    ,'game_ot'
    ,'game_attendance'
    ,'game_length'
    ,'game_notes'
]
df

,game_date,game_time,team_away,team_away_goals,team_home,team_home_goals,game_ot,game_attendance,game_length,game_notes
0,2025-10-07,5:00 PM,Chicago Blackhawks,2.0,Florida Panthers,3.0,NaN,19655.0,2:34,NaN
1,2025-10-07,10:30 PM,Colorado Avalanche,4.0,Los Angeles Kings,1.0,NaN,18145.0,2:37,NaN
2,2025-10-07,8:00 PM,Pittsburgh Penguins,3.0,New York Rangers,0.0,NaN,18006.0,2:16,NaN
3,2025-10-08,10:00 PM,Calgary Flames,4.0,Edmonton Oilers,3.0,SO,18347.0,2:55,NaN
4,2025-10-08,7:00 PM,Montreal Canadiens,2.0,Toronto Maple Leafs,5.0,NaN,19037.0,2:25,NaN
...,...,...,...,...,...,...,...,...,...,...
1307,2026-04-16,10:30 PM,Seattle Kraken,NaN,Colorado Avalanche,NaN,NaN,NaN,NaN,NaN
1308,2026-04-16,9:00 PM,Vancouver Canucks,NaN,Edmonton Oilers,NaN,NaN,NaN,NaN,NaN
1309,2026-04-16,8:00 PM,Anaheim Ducks,NaN,Nashville Predators,NaN,NaN,NaN,NaN,NaN
1310,2026-04-16,8:00 PM,St. Louis Blues,NaN,Utah Mammoth,NaN,NaN,NaN,NaN,NaN


In [8]:
def winning_team_test(team_a, score_a, team_b, score_b, ot_result):
    if score_a > score_b and pd.isna(ot_result) == False:
        return team_a, 2, team_b, 1
    elif score_a > score_b:
        return team_a, 2, team_b, 0
    elif score_b > score_a and pd.isna(ot_result) == False:
        return team_b, 2, team_a, 1
    elif score_b > score_a:
        return team_b, 2, team_a, 0

In [9]:
winning_team_test('sabres',3,'leafs',1,'OT')

('sabres', 2, 'leafs', 1)

In [10]:
def winning_team(row):
    if row['team_away_goals'] > row['team_home_goals'] and pd.isna(row['game_ot']) == False:
        return row['team_away'], 2, row['team_home'], 1
    elif row['team_away_goals'] > row['team_home_goals']:
        return row['team_away'], 2, row['team_home'], 0
    elif row['team_home_goals'] > row['team_away_goals'] and pd.isna(row['game_ot']) == False:
        return row['team_home'], 2, row['team_away'], 1
    elif row['team_home_goals'] > row['team_away_goals']:
        return row['team_home'], 2, row['team_away'], 0
    else:
        return 'future_game', 0, 'future_game', 0

In [11]:
df['game_winning_team'] = df.apply(lambda row: winning_team(row)[0], axis=1)
df['game_winning_points'] = df.apply(lambda row: winning_team(row)[1], axis=1)
df['game_losing_team'] = df.apply(lambda row: winning_team(row)[2], axis=1)
df['game_losing_points'] = df.apply(lambda row: winning_team(row)[3], axis=1)

In [12]:
df

,game_date,game_time,team_away,team_away_goals,team_home,team_home_goals,game_ot,game_attendance,game_length,game_notes,game_winning_team,game_winning_points,game_losing_team,game_losing_points
0,2025-10-07,5:00 PM,Chicago Blackhawks,2.0,Florida Panthers,3.0,NaN,19655.0,2:34,NaN,Florida Panthers,2,Chicago Blackhawks,0
1,2025-10-07,10:30 PM,Colorado Avalanche,4.0,Los Angeles Kings,1.0,NaN,18145.0,2:37,NaN,Colorado Avalanche,2,Los Angeles Kings,0
2,2025-10-07,8:00 PM,Pittsburgh Penguins,3.0,New York Rangers,0.0,NaN,18006.0,2:16,NaN,Pittsburgh Penguins,2,New York Rangers,0
3,2025-10-08,10:00 PM,Calgary Flames,4.0,Edmonton Oilers,3.0,SO,18347.0,2:55,NaN,Calgary Flames,2,Edmonton Oilers,1
4,2025-10-08,7:00 PM,Montreal Canadiens,2.0,Toronto Maple Leafs,5.0,NaN,19037.0,2:25,NaN,Toronto Maple Leafs,2,Montreal Canadiens,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1307,2026-04-16,10:30 PM,Seattle Kraken,NaN,Colorado Avalanche,NaN,NaN,NaN,NaN,NaN,future_game,0,future_game,0
1308,2026-04-16,9:00 PM,Vancouver Canucks,NaN,Edmonton Oilers,NaN,NaN,NaN,NaN,NaN,future_game,0,future_game,0
1309,2026-04-16,8:00 PM,Anaheim Ducks,NaN,Nashville Predators,NaN,NaN,NaN,NaN,NaN,future_game,0,future_game,0
1310,2026-04-16,8:00 PM,St. Louis Blues,NaN,Utah Mammoth,NaN,NaN,NaN,NaN,NaN,future_game,0,future_game,0


In [13]:
df_winning_team = df[['game_date','game_winning_team', 'game_winning_points']]
df_winning_team.columns = ['game_date', 'team', 'points']

df_losing_team = df[['game_date','game_losing_team', 'game_losing_points']]
df_losing_team.columns = ['game_date', 'team', 'points']

In [14]:
df_team_game_results = pd.concat([df_winning_team, df_losing_team], ignore_index=True)

In [15]:
df_team_game_results

,game_date,team,points
0,2025-10-07,Florida Panthers,2
1,2025-10-07,Colorado Avalanche,2
2,2025-10-07,Pittsburgh Penguins,2
3,2025-10-08,Calgary Flames,2
4,2025-10-08,Toronto Maple Leafs,2
...,...,...,...
2619,2026-04-16,future_game,0
2620,2026-04-16,future_game,0
2621,2026-04-16,future_game,0
2622,2026-04-16,future_game,0


In [16]:
atlantic_division = [
    'Buffalo Sabres'
    ,'Tampa Bay Lightning'
    ,'Montreal Canadiens'
    ,'Boston Bruins'
    ,'Detroit Red Wings'
    ,'Ottawa Senators'
    ,'Toronto Maple Leafs'
    ,'Florida Panthers'
]

In [17]:
df_atlantic_division = df_team_game_results[df_team_game_results['team'].isin(atlantic_division)].reset_index(drop=True)

In [18]:
df_atlantic_division

,game_date,team,points
0,2025-10-07,Florida Panthers,2
1,2025-10-08,Toronto Maple Leafs,2
2,2025-10-08,Boston Bruins,2
3,2025-10-09,Boston Bruins,2
4,2025-10-09,Montreal Canadiens,2
...,...,...,...
526,2026-03-14,Detroit Red Wings,1
527,2026-03-14,Montreal Canadiens,0
528,2026-03-14,Tampa Bay Lightning,0
529,2026-03-15,Montreal Canadiens,0


In [19]:
month_abbr = dict(enumerate(calendar.month_abbr))
df_atlantic_division['month_num'] = df_atlantic_division['game_date'].str[5:7].astype(int)
df_atlantic_division['month_num_sort'] = df_atlantic_division['game_date'].str[2:4] + '-' + df_atlantic_division['game_date'].str[5:7]
df_atlantic_division['month'] = df_atlantic_division['month_num'].map(month_abbr)
df_atlantic_division

,game_date,team,points,month_num,month_num_sort,month
0,2025-10-07,Florida Panthers,2,10,25-10,Oct
1,2025-10-08,Toronto Maple Leafs,2,10,25-10,Oct
2,2025-10-08,Boston Bruins,2,10,25-10,Oct
3,2025-10-09,Boston Bruins,2,10,25-10,Oct
4,2025-10-09,Montreal Canadiens,2,10,25-10,Oct
...,...,...,...,...,...,...
526,2026-03-14,Detroit Red Wings,1,3,26-03,Mar
527,2026-03-14,Montreal Canadiens,0,3,26-03,Mar
528,2026-03-14,Tampa Bay Lightning,0,3,26-03,Mar
529,2026-03-15,Montreal Canadiens,0,3,26-03,Mar


In [20]:
df_points_by_month = df_atlantic_division.groupby(['team', 'month', 'month_num', 'month_num_sort'], as_index=False).agg(
    month_points=('points','sum')
).sort_values(by='month_num_sort').reset_index(drop=True)

In [22]:
df_points_by_month['running_points'] = df_points_by_month.groupby('team')['month_points'].cumsum()
df_points_by_month['standing_by_month'] = df_points_by_month.groupby('month_num_sort')['running_points'].rank(method='dense', ascending=False).astype(int)
df_points_by_month

,team,month,month_num,month_num_sort,month_points,running_points,standing_by_month
0,Florida Panthers,Oct,10,25-10,11,11,4
1,Tampa Bay Lightning,Oct,10,25-10,12,12,3
2,Ottawa Senators,Oct,10,25-10,13,13,2
3,Montreal Canadiens,Oct,10,25-10,16,16,1
4,Detroit Red Wings,Oct,10,25-10,16,16,1
5,Buffalo Sabres,Oct,10,25-10,11,11,4
6,Toronto Maple Leafs,Oct,10,25-10,11,11,4
7,Boston Bruins,Oct,10,25-10,12,12,3
8,Ottawa Senators,Nov,11,25-11,15,28,4
9,Buffalo Sabres,Nov,11,25-11,13,24,6


In [28]:
def assign_colors(row):
    if row['team'] == 'Toronto Maple Leafs':
        return '#BBCDF0'
    elif row['team'] == 'Ottawa Senators':
        return '#FACAF8'
    elif row['team'] == 'Tampa Bay Lightning':
        return '#DBD5D5'
    elif row['team'] == 'Florida Panthers':
        return '#DFC3F7'
    elif row['team'] == 'Detroit Red Wings':
        return '#F78895'
    elif row['team'] == 'Boston Bruins':
        return '#FCDC95'
    elif row['team'] == 'Montreal Canadiens':
        return '#C1F5E0'
    elif row['team'] == 'Buffalo Sabres':
        return '#000000'
    else:
        return '#E62727'

In [29]:
df_points_by_month['team_color'] = df_points_by_month.apply(assign_colors, axis=1)
df_points_by_month

,team,month,month_num,month_num_sort,month_points,running_points,standing_by_month,team_color
0,Florida Panthers,Oct,10,25-10,11,11,4,#DFC3F7
1,Tampa Bay Lightning,Oct,10,25-10,12,12,3,#DBD5D5
2,Ottawa Senators,Oct,10,25-10,13,13,2,#FACAF8
3,Montreal Canadiens,Oct,10,25-10,16,16,1,#C1F5E0
4,Detroit Red Wings,Oct,10,25-10,16,16,1,#F78895
5,Buffalo Sabres,Oct,10,25-10,11,11,4,#000000
6,Toronto Maple Leafs,Oct,10,25-10,11,11,4,#BBCDF0
7,Boston Bruins,Oct,10,25-10,12,12,3,#FCDC95
8,Ottawa Senators,Nov,11,25-11,15,28,4,#FACAF8
9,Buffalo Sabres,Nov,11,25-11,13,24,6,#000000
